In [7]:
#imports cels
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
# sample dataset (you can replace with kaggle file later)
data = {
    "review": [
        "I love this movie",
        "This film is terrible",
        "Amazing acting and story",
        "Worst movie ever",
        "I really enjoyed it",
        "Not good at all"
    ],
    "sentiment": ["positive","negative","positive","negative","positive","negative"]
}

df = pd.DataFrame(data)
df.head()

,review,sentiment
0,I love this movie,positive
1,This film is terrible,negative
2,Amazing acting and story,positive
3,Worst movie ever,negative
4,I really enjoyed it,positive


In [9]:
print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df['sentiment'].value_counts())
print("\nSample Data:")
print(df.head())

Dataset Shape: (6, 2)

Class Distribution:
sentiment
positive    3
negative    3
Name: count, dtype: int64

Sample Data:
                     review sentiment
0         I love this movie  positive
1     This film is terrible  negative
2  Amazing acting and story  positive
3          Worst movie ever  negative
4       I really enjoyed it  positive


In [10]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)
df['clean_text'] = df['review'].apply(preprocess_text)
df.head()

,review,sentiment,clean_text
0,I love this movie,positive,love movi
1,This film is terrible,negative,film terribl
2,Amazing acting and story,positive,amaz act stori
3,Worst movie ever,negative,worst movi ever
4,I really enjoyed it,positive,realli enjoy


In [11]:
# Feature Engineering (BoW) and Tf-IDF
bow = CountVectorizer()
X_bow = bow.fit_transform(df['clean_text'])
y = df['sentiment']

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])

In [12]:
# Train Test Split and Logistic Regression
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [13]:
# Naive Bayes and Decision Tree
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [14]:
# Evaluation Function
def evaluate_model(y_true, y_pred, model_name):
    print("Model:", model_name)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, pos_label="positive"))
    print("Recall:", recall_score(y_true, y_pred, pos_label="positive"))
    print("F1 Score:", f1_score(y_true, y_pred, pos_label="positive"))
    print("-"*50)

In [15]:
# Evaluate All Models
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model: Logistic Regression
Accuracy: 0.5
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
--------------------------------------------------
Model: Naive Bayes
Accuracy: 0.5
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
--------------------------------------------------
Model: Decision Tree
Accuracy: 0.5
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Logistic Regression performed best.
Naive Bayes worked well for text classification.
Decision Tree showed lower performance.
TF-IDF gave better results than Bag of Words.